In [1]:
import sys
from pathlib import Path
import os

# --- Robustly locate repo root by searching upward for a folder named "src" ---
cwd = Path.cwd().resolve()
repo_root = None

for p in [cwd] + list(cwd.parents):
    if (p / "src").is_dir():
        repo_root = p
        break

if repo_root is None:
    raise RuntimeError(f"Could not find repo root containing 'src/' starting from {cwd}")

# Add repo root to sys.path so `import src...` works
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("cwd:", os.getcwd())
print("repo_root:", repo_root)
print("sys.path[0]:", sys.path[0])

# --- Now imports from src will work ---
import numpy as np
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms.optimizers import COBYLA, SPSA
from src.vqe_utils import make_two_local_ansatz, run_vqe_exact, repeat_vqe_finite_shots

# --- VQE ---
H = SparsePauliOp.from_list([("ZZ", 1.0), ("XI", 0.5), ("IX", 0.5)])
ansatz = make_two_local_ansatz(num_qubits=2, reps=1)

E_exact = run_vqe_exact(operator=H, ansatz=ansatz, maxiter=100)

cobyla_summary = repeat_vqe_finite_shots(
    operator=H,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=100),
    shots=256,
    num_runs=3,
)

spsa_summary = repeat_vqe_finite_shots(
    operator=H,
    ansatz=ansatz,
    optimizer=SPSA(maxiter=100),
    shots=256,
    num_runs=3,
)

print("Exact:", E_exact)
print("COBYLA mean±std:", cobyla_summary.mean, cobyla_summary.std)
print("SPSA   mean±std:", spsa_summary.mean, spsa_summary.std)

cwd: /Users/bjr/Library/CloudStorage/Dropbox/QC/quantum-algorithms-lab/notebooks
repo_root: /Users/bjr/Library/CloudStorage/Dropbox/QC/quantum-algorithms-lab
sys.path[0]: /Users/bjr/Library/CloudStorage/Dropbox/QC/quantum-algorithms-lab
Exact: -1.403021686793207
COBYLA mean±std: -1.3719071751429401 0.05982953538910319
SPSA   mean±std: -1.3789246888793414 0.039816407060895
